# Классификация автоматизированного трафика Авито

**Задача**: по событиям внутри суточного окна оценить вероятность того, что `cookie_id` принадлежит трафику известных сервисов сбора данных. Основная метрика - максимальный precision при recall не ниже 70% на уровне куки.

Импортируем необходимые библиотеки. Фиксируем константы и SEED.

In [541]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from metric import precision_at_recall
from catboost import CatBoostClassifier
from time import perf_counter


DATA_DIR = Path('data')
SEED = 42
np.random.seed(SEED)

## 1. Проверка данных и временного окна

Сначала проверяем схему, уникальность куки и состав примера ответа. Затем оставляем только события с `window_start_ts <= event_ts < window_end_ts`. Полные дубликаты удаляются после фильтрации времени, разные события в одну секунду сохраняются.

Загружаем данные

In [431]:
date_cols = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=date_cols)
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=date_cols)
events = pd.read_csv(DATA_DIR / 'events.csv', parse_dates=['event_ts'])
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

In [432]:
train.head()

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [433]:
test.head()

,cookie_id,cookie_created_at,window_start_ts,window_end_ts
0,ck_315fb710a0e371e7,2026-02-20 08:52:31,2026-04-20,2026-04-21
1,ck_a76ee3b3e3e522fd,2026-01-19 13:42:15,2026-04-20,2026-04-21
2,ck_94c9a4d382689e82,2026-04-19 06:28:37,2026-04-20,2026-04-21
3,ck_8eaf9509ad9462a0,2026-01-19 17:18:06,2026-04-20,2026-04-21
4,ck_9a88a5a989cb5bc6,2025-11-09 17:25:13,2026-04-20,2026-04-21


In [434]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


In [435]:
sample_submission.head()

,cookie_id,score
0,ck_99a4e5ef02d89493,0.5
1,ck_54d463f7fd89ec3d,0.5
2,ck_0fbbc7a6af300368,0.5
3,ck_8fd4937eadd64fb6,0.5
4,ck_dbb4bd4eda97255d,0.5


Минимальная проверка данных

In [436]:
print(f"Проверка на уникальность записей. Train: {train.cookie_id.is_unique}. Test: {test.cookie_id.is_unique}")
print(f"Куки из train и test пересекаются? - {"Нет" if set(train.cookie_id).isdisjoint(test.cookie_id) else "Да"}")
print(f"Время начала окна меньше, чем время конца в Train? - {(train.window_start_ts < train.window_end_ts).all()}")
print(f"Время начала окна меньше, чем время конца в Test? - {(test.window_start_ts < test.window_end_ts).all()}")

# Объединим train и test в один датасет
meta = pd.concat([train, test], ignore_index=True)

Проверка на уникальность записей. Train: True. Test: True
Куки из train и test пересекаются? - Нет
Время начала окна меньше, чем время конца в Train? - True
Время начала окна меньше, чем время конца в Test? - True


In [437]:
# Проверяем уникальность куки
assert meta.cookie_id.is_unique
# Проверяем, чтобы куки из events были в объединенном датасете meta
assert set(events.cookie_id).issubset(meta.cookie_id)

In [438]:
# Берем только те данные, которые входят в окно наблюдения
windowed = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id', validate='many_to_one')
windowed = windowed.loc[(windowed.event_ts >= windowed.window_start_ts) & (windowed.event_ts < windowed.window_end_ts)]

windowed.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN,2026-04-26,2026-04-27
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN,2026-04-20,2026-04-21
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0,2026-04-13,2026-04-14
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN,2026-04-08,2026-04-09
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN,2026-04-12,2026-04-13


In [439]:
# Удаляем полные дубликаты
clean_events = windowed.drop_duplicates(subset=events.columns.tolist()).copy()

# Форматируем признак platform, так как значения "desktop", "Web", "web" и "WEB" по сути равны.
# То же самое и с android и ios.
clean_events['platform'] = (
    clean_events.platform.str.casefold()
    .replace({'desktop': 'web', 'iphone': 'ios'})
)

# Сортируем значения по куки и событиям.
clean_events = clean_events.sort_values(
    ['cookie_id', 'event_ts'], kind='stable'
).reset_index(drop=True)

clean_events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
0,ck_00013fdfa0bd37dd,2026-04-24 18:41:16,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1012349.0,rabota,moskva,private,NaN,NaN,NaN,NaN,2026-04-24,2026-04-25
1,ck_00013fdfa0bd37dd,2026-04-24 19:16:53,200,item_view,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1054355.0,rabota,moskva,private,NaN,NaN,NaN,NaN,2026-04-24,2026-04-25
2,ck_00013fdfa0bd37dd,2026-04-24 19:16:54,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1015683.0,detskie_tovary,habarovsk,pro,NaN,NaN,NaN,NaN,2026-04-24,2026-04-25
3,ck_00013fdfa0bd37dd,2026-04-24 19:17:05,200,item_view,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1028622.0,avtomobili,moskva,private,NaN,NaN,NaN,NaN,2026-04-24,2026-04-25
4,ck_00013fdfa0bd37dd,2026-04-24 20:05:05,500,login,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-24,2026-04-25


In [440]:
summary = pd.Series({
    'train cookies': len(train),
    'test cookies': len(test),
    'positive train cookies': int(train.target.sum()),
    'all events': len(events),
    'events inside window': len(windowed),
    'full duplicates inside window': len(windowed) - len(clean_events),
    'clean events': len(clean_events),
})
display(summary.to_frame('count'))

,count
train cookies,11091
test cookies,4909
positive train cookies,899
all events,328905
events inside window,288126
full duplicates inside window,4293
clean events,283833


In [441]:
# Проверка на вхождение в интервал окна наблюдения.
assert clean_events.event_ts.ge(clean_events.window_start_ts).all()
assert clean_events.event_ts.lt(clean_events.window_end_ts).all()
assert clean_events.event_name.ne('captcha_shown').all()

### Вывод по этапу №1:

Для проверки моделей используем временные разбиения по дате начала окна window_start_ts. 
- На A обучаемся на 6-9 апреля и проверяем на 10-12 апреля; 
- На B обучаемся на 6-12 апреля и проверяем на 13-16 апреля; 
- На C обучаемся на 6-16 апреля и проверяем на 17-19 апреля. 

Это позволяет оценивать качество на более поздних куки, не подмешивая их в обучение.

## 2. Baseline

Воспроизводим quickstart: два признака (`n_events`, `item_nunique`) и RandomForest с 300 деревьями и `min_samples_leaf=3`. Обучение: 6-16 апреля; проверка: 17-19 апреля. Метрика берётся из `metric.py`. Прогнозы сохраняются в `artifacts/baseline/`, строка результата - в `reports/experiments.csv`.

In [442]:
# Агрегируем данные по кукам.
baseline_features = (
    windowed.groupby('cookie_id').agg(
        n_events=('event_ts', 'size'),
        item_nunique=('item_id', 'nunique'),
    ).reset_index()
)

# Создаем фрейм из двух признака: количество событий и количество уникальных объявлений.
X_baseline = (
    train[['cookie_id']]
    .merge(baseline_features, on='cookie_id', how='left', validate='one_to_one')
    .fillna({'n_events': 0, 'item_nunique': 0})
)

X_baseline.head()

,cookie_id,n_events,item_nunique
0,ck_54a059eb7d3ea68b,7,4
1,ck_7e4de46eeab82974,41,19
2,ck_9320229ef6304522,36,19
3,ck_30ccd25bc1714ed9,21,10
4,ck_a77c5f05948cdeef,29,16


In [443]:
feature_cols = ['n_events', 'item_nunique']
split_date = pd.Timestamp('2026-04-17')
is_train = train.window_start_ts < split_date

In [444]:
# Создаем базовую модель и обучаем ее на нашем X_baseline
baseline = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=3, random_state=SEED
)
baseline.fit(X_baseline.loc[is_train, feature_cols], train.loc[is_train, 'target'])
y_valid = train.loc[~is_train, 'target']
score_valid = baseline.predict_proba(X_baseline.loc[~is_train, feature_cols])[:, 1]

In [445]:
# Сохраняем результат в словарик. 
baseline_metrics = {
    'experiment': 'quickstart_baseline_C',
    'train_before': str(split_date.date()),
    'valid_from': str(split_date.date()),
    'n_train': int(is_train.sum()),
    'n_valid': int((~is_train).sum()),
    'n_valid_positive': int(y_valid.sum()),
    'precision_at_recall_70': precision_at_recall(y_valid, score_valid),
    'average_precision': average_precision_score(y_valid, score_valid),
    'roc_auc': roc_auc_score(y_valid, score_valid),
    'random_state': SEED,
    'deduplicate': False,
}

# Сохраняем результат baseline в ./artifacts и ставим запись в reports/experiments.csv
predictions = train.loc[~is_train, ['cookie_id', 'target']].copy()
predictions['score'] = score_valid

Path('artifacts/baseline').mkdir(parents=True, exist_ok=True)
predictions.to_csv('artifacts/baseline/validation_C.csv', index=False)
report_path = Path('reports/experiments.csv')
report_path.parent.mkdir(parents=True, exist_ok=True)

In [446]:
previous = pd.read_csv(report_path) if report_path.exists() else pd.DataFrame()
if not previous.empty:
    previous = previous.loc[previous.experiment.ne(baseline_metrics['experiment'])]
pd.concat([previous, pd.DataFrame([baseline_metrics])], ignore_index=True).to_csv(
    report_path, index=False
)
display(pd.Series(baseline_metrics).to_frame('value'))

,value
experiment,quickstart_baseline_C
train_before,2026-04-17
valid_from,2026-04-17
n_train,9140
n_valid,1951
n_valid_positive,160
precision_at_recall_70,0.106161
average_precision,0.180385
roc_auc,0.608129
random_state,42


При seed=42 результат на проверке 17–19 апреля: **P@R≥0.70 = 0.106161**, Average Precision = 0.180385, ROC-AUC = 0.608129.

## 3. Поведенческие признаки

Собираем одну строку на куку только из `clean_events` внутри её окна. Признаки группировал так: состав действий, время, сессии, устройство и поиск. Целевая переменная в расчёт не передаётся.

In [447]:
# Общая таблица: одна строка на cookie_id, без target.
features = meta[['cookie_id']].set_index('cookie_id')
cookie_id_groupby = clean_events.groupby('cookie_id', sort=False)
features['n_events'] = cookie_id_groupby.size()

features

,n_events
cookie_id,
ck_54a059eb7d3ea68b,7
ck_7e4de46eeab82974,41
ck_9320229ef6304522,36
ck_30ccd25bc1714ed9,21
ck_a77c5f05948cdeef,28
...,...
ck_8b772caed8f1541c,2
ck_5403c76fe68a08ba,22
ck_3e6833574edf1736,9


In [448]:
# Сколько раз встретился каждый тип действия и какова его доля.
event_counts = pd.crosstab(clean_events.cookie_id, clean_events.event_name)
event_counts

event_name,contact_chat_open,contact_message_sent,contact_phone_show,favorite_add,item_view,login,photo_swipe,search_results_view,seller_page_view
cookie_id,,,,,,,,,
ck_00013fdfa0bd37dd,2,0,0,1,4,2,3,3,0
ck_000c95f1408dcb00,0,0,0,0,11,0,1,3,1
ck_000e8c52636e3bec,0,0,1,4,11,1,5,9,3
ck_0010e31baa4a1fb7,0,0,0,2,7,1,1,4,1
ck_0010ec3874fb5378,0,0,0,10,25,1,14,20,4
...,...,...,...,...,...,...,...,...,...
ck_ffee36a37e2532f5,1,2,6,1,25,0,14,24,5
ck_fff88a7920fd0fb6,0,0,0,0,7,0,1,2,0
ck_fff9fdc2adf5100b,0,1,7,0,17,2,4,2,2


In [449]:
for event_name in sorted(clean_events.event_name.unique()):
    features[f'n_{event_name}'] = event_counts[event_name]
    features[f'share_{event_name}'] = features[f'n_{event_name}'] / features.n_events
features

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,share_item_view,n_login,share_login,n_photo_swipe,share_photo_swipe,n_search_results_view,share_search_results_view,n_seller_page_view,share_seller_page_view
cookie_id,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,0.428571,0,0.000000,1,0.142857,3,0.428571,0,0.000000
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,0.536585,1,0.024390,4,0.097561,8,0.195122,4,0.097561
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,0.194444,3,0.083333,3,0.083333,13,0.361111,3,0.083333
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,0.666667,0,0.000000,0,0.000000,3,0.142857,2,0.095238
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,0.428571,0,0.000000,4,0.142857,6,0.214286,3,0.107143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ck_8b772caed8f1541c,2,0,0.000000,0,0.000000,0,0.000000,0,0.000000,2,1.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000
ck_5403c76fe68a08ba,22,0,0.000000,0,0.000000,0,0.000000,0,0.000000,12,0.545455,0,0.000000,3,0.136364,6,0.272727,1,0.045455
ck_3e6833574edf1736,9,0,0.000000,0,0.000000,0,0.000000,1,0.111111,2,0.222222,1,0.111111,1,0.111111,3,0.333333,1,0.111111


Возраст куки, число заполненных полей и разные интересы могут отличать массовый сбор объявлений от обычного просмотра. Доли считаем относительно числа наблюдаемых значений, чтобы пропуск не выглядел как отдельное действие.

In [450]:
meta_by_cookie = meta.set_index('cookie_id')
features['cookie_age_start_days'] = (meta_by_cookie.window_start_ts - meta_by_cookie.cookie_created_at).dt.total_seconds() / 86400
features['cookie_age_end_days'] = (meta_by_cookie.window_end_ts - meta_by_cookie.cookie_created_at).dt.total_seconds() / 86400
features['created_in_window'] = (meta_by_cookie.cookie_created_at >= meta_by_cookie.window_start_ts).astype(int)
features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,share_login,n_photo_swipe,share_photo_swipe,n_search_results_view,share_search_results_view,n_seller_page_view,share_seller_page_view,cookie_age_start_days,cookie_age_end_days,created_in_window
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,0.000000,1,0.142857,3,0.428571,0,0.000000,135.603692,136.603692,0
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,0.024390,4,0.097561,8,0.195122,4,0.097561,194.576111,195.576111,0
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,0.083333,3,0.083333,13,0.361111,3,0.083333,32.994421,33.994421,0
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,0.000000,0,0.000000,3,0.142857,2,0.095238,0.546759,1.546759,0
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,0.000000,4,0.142857,6,0.214286,3,0.107143,94.837095,95.837095,0
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,0.000000,2,0.111111,7,0.388889,0,0.000000,36.264525,37.264525,0
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,0.000000,3,0.300000,1,0.100000,1,0.100000,154.321470,155.321470,0
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,0.000000,1,0.043478,6,0.260870,1,0.043478,0.337708,1.337708,0
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,0.000000,0,0.000000,2,0.500000,0,0.000000,243.436227,244.436227,0


In [451]:
# Отдельно храним объём и разнообразие: один просмотр не равен сотне.
for column in ['item_id', 'item_category', 'item_location',
               'search_query', 'seller_type', 'user_agent', 'platform']:
    features[f'n_{column}_present'] = cookie_id_groupby[column].count()
    features[f'n_{column}_unique'] = cookie_id_groupby[column].nunique()
    features[f'unique_{column}_share'] = features[f'n_{column}_unique'] / features[f'n_{column}_present'].replace(0, np.nan)

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,unique_search_query_share,n_seller_type_present,n_seller_type_unique,unique_seller_type_share,n_user_agent_present,n_user_agent_unique,unique_user_agent_share,n_platform_present,n_platform_unique,unique_platform_share
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,1.000000,4,2,0.500000,7,1,0.142857,7,1,0.142857
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,0.625000,28,2,0.071429,41,1,0.024390,41,1,0.024390
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,0.769231,18,2,0.111111,36,1,0.027778,36,1,0.027778
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,1.000000,18,2,0.111111,21,1,0.047619,21,1,0.047619
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,1.000000,20,2,0.100000,28,1,0.035714,28,1,0.035714
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,0.857143,9,2,0.222222,18,1,0.055556,18,1,0.055556
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,1.000000,9,2,0.222222,10,1,0.100000,10,1,0.100000
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,0.500000,17,2,0.117647,23,1,0.043478,23,1,0.043478
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,1.000000,1,1,1.000000,4,1,0.250000,4,1,0.250000


In [452]:
# Сколько просмотрено объявлений, количество повторных просмотров
item_views = clean_events.loc[clean_events.event_name == 'item_view']
features['n_viewed_items'] = item_views.groupby('cookie_id').item_id.nunique().fillna(0)
features['repeat_item_view_share'] = (features.n_item_view - features.n_viewed_items) / features.n_item_view.replace(0, np.nan)

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,n_seller_type_unique,unique_seller_type_share,n_user_agent_present,n_user_agent_unique,unique_user_agent_share,n_platform_present,n_platform_unique,unique_platform_share,n_viewed_items,repeat_item_view_share
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,2,0.500000,7,1,0.142857,7,1,0.142857,3.0,0.000000
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,2,0.071429,41,1,0.024390,41,1,0.024390,14.0,0.363636
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,2,0.111111,36,1,0.027778,36,1,0.027778,6.0,0.142857
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,2,0.111111,21,1,0.047619,21,1,0.047619,9.0,0.357143
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,2,0.100000,28,1,0.035714,28,1,0.035714,12.0,0.000000
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,2,0.222222,18,1,0.055556,18,1,0.055556,4.0,0.000000
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,2,0.222222,10,1,0.100000,10,1,0.100000,3.0,0.000000
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,2,0.117647,23,1,0.043478,23,1,0.043478,5.0,0.444444
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,1,1.000000,4,1,0.250000,4,1,0.250000,1.0,0.000000


Время измеряем относительно границ окна. Выдклим количество активных часов и плотность событий в одной минуте.

In [453]:
# Первое и последнее события для куки в окне.
first_event = cookie_id_groupby.event_ts.min()
last_event = cookie_id_groupby.event_ts.max()
# Активное окно событий
features['activity_span_s'] = (last_event - first_event).dt.total_seconds()
features['first_event_delay_s'] = (first_event - meta_by_cookie.window_start_ts).dt.total_seconds()
features['last_event_before_end_s'] = (meta_by_cookie.window_end_ts - last_event).dt.total_seconds()

# Активные часы и минуты для кукисов)
features['active_hours'] = clean_events.groupby('cookie_id').event_ts.agg(lambda s: s.dt.floor('h').nunique())
features['active_minutes'] = clean_events.groupby('cookie_id').event_ts.agg(lambda s: s.dt.floor('min').nunique())

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,n_platform_present,n_platform_unique,unique_platform_share,n_viewed_items,repeat_item_view_share,activity_span_s,first_event_delay_s,last_event_before_end_s,active_hours,active_minutes
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,7,1,0.142857,3.0,0.000000,167.0,5487.0,80746.0,1,4
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,41,1,0.024390,14.0,0.363636,39476.0,39026.0,7898.0,8,30
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,36,1,0.027778,6.0,0.142857,36491.0,4254.0,45655.0,7,21
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,21,1,0.047619,9.0,0.357143,17221.0,8850.0,60329.0,3,16
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,28,1,0.035714,12.0,0.000000,19098.0,29279.0,38023.0,5,25
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,18,1,0.055556,4.0,0.000000,1054.0,39102.0,46244.0,2,11
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,10,1,0.100000,3.0,0.000000,6849.0,2733.0,76818.0,2,7
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,23,1,0.043478,5.0,0.444444,19840.0,63705.0,2855.0,5,16
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,4,1,0.250000,1.0,0.000000,95.0,81746.0,4559.0,1,3


In [454]:
# Количество действий в минуту и доля действий в минуту от всех действий
minute_counts = clean_events.groupby(['cookie_id', clean_events.event_ts.dt.floor('min')]).size()
features['max_events_per_minute'] = minute_counts.groupby(level=0).max()
features['top_minute_share'] = features.max_events_per_minute / features.n_events

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,unique_platform_share,n_viewed_items,repeat_item_view_share,activity_span_s,first_event_delay_s,last_event_before_end_s,active_hours,active_minutes,max_events_per_minute,top_minute_share
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,0.142857,3.0,0.000000,167.0,5487.0,80746.0,1,4,2,0.285714
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,0.024390,14.0,0.363636,39476.0,39026.0,7898.0,8,30,3,0.073171
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,0.027778,6.0,0.142857,36491.0,4254.0,45655.0,7,21,4,0.111111
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,0.047619,9.0,0.357143,17221.0,8850.0,60329.0,3,16,2,0.095238
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,0.035714,12.0,0.000000,19098.0,29279.0,38023.0,5,25,2,0.071429
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,0.055556,4.0,0.000000,1054.0,39102.0,46244.0,2,11,3,0.166667
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,0.100000,3.0,0.000000,6849.0,2733.0,76818.0,2,7,2,0.200000
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,0.043478,5.0,0.444444,19840.0,63705.0,2855.0,5,16,3,0.130435
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,0.250000,1.0,0.000000,95.0,81746.0,4559.0,1,3,2,0.500000


Теперь посчитаем паузы между действиями. Интервалы считаем после сортировки событий внутри куки. Нулевые интервалы выделяем отдельно: при одинаковой секунде точный порядок действий неизвестен. Для куки с одним событием статистика интервалов остаётся пропущенной.

In [455]:
# Считаем паузы между дейсвтиями
clean_events['gap_s'] = cookie_id_groupby.event_ts.diff().dt.total_seconds()
# Добавляем статистики по паузам
gap_stats = cookie_id_groupby.gap_s.agg(['median', 'mean', 'std', 'min', 'max'])
for stat in gap_stats.columns:
    features[f'gap_{stat}_s'] = gap_stats[stat]
features['gap_q10_s'] = cookie_id_groupby.gap_s.quantile(0.10)
features['gap_q90_s'] = cookie_id_groupby.gap_s.quantile(0.90)
features['gap_cv'] = features.gap_std_s / features.gap_mean_s.replace(0, np.nan)
features['n_gaps'] = cookie_id_groupby.gap_s.count()

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,top_minute_share,gap_median_s,gap_mean_s,gap_std_s,gap_min_s,gap_max_s,gap_q10_s,gap_q90_s,gap_cv,n_gaps
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,0.285714,21.0,27.833333,29.949402,1.0,84.0,3.5,59.0,1.076026,6
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,0.073171,57.5,986.900000,2416.286861,2.0,8511.0,6.9,3889.7,2.448360,40
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,0.111111,22.0,1042.600000,2306.449400,1.0,8350.0,4.8,4513.4,2.212209,35
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,0.095238,54.5,861.050000,2502.081501,6.0,8604.0,17.9,877.9,2.905849,20
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,0.071429,52.0,707.333333,1748.439172,22.0,7224.0,28.6,2743.2,2.471874,27
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,0.166667,32.0,62.000000,57.989223,8.0,174.0,14.0,148.4,0.935310,17
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,0.200000,80.0,761.000000,2023.827871,1.0,6151.0,1.0,1490.2,2.659432,9
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,0.130435,35.0,901.818182,2169.186454,9.0,8140.0,12.7,1618.0,2.405348,22
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,0.500000,11.0,31.666667,40.203648,6.0,78.0,7.0,64.6,1.269589,3


In [456]:
# Частые короткие паузы и точные совпадения времени считаем отдельно.
for seconds in [0, 1, 5, 10]:
    short = clean_events.gap_s.eq(0) if seconds == 0 else clean_events.gap_s.le(seconds)
    features[f'gap_le_{seconds}_share'] = (
        short.groupby(clean_events.cookie_id).sum()
        / features.n_gaps.replace(0, np.nan)
    )

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,gap_min_s,gap_max_s,gap_q10_s,gap_q90_s,gap_cv,n_gaps,gap_le_0_share,gap_le_1_share,gap_le_5_share,gap_le_10_share
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,1.0,84.0,3.5,59.0,1.076026,6,0.0,0.166667,0.166667,0.333333
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,2.0,8511.0,6.9,3889.7,2.448360,40,0.0,0.000000,0.075000,0.200000
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,1.0,8350.0,4.8,4513.4,2.212209,35,0.0,0.028571,0.114286,0.171429
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,6.0,8604.0,17.9,877.9,2.905849,20,0.0,0.000000,0.000000,0.050000
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,22.0,7224.0,28.6,2743.2,2.471874,27,0.0,0.000000,0.000000,0.000000
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,8.0,174.0,14.0,148.4,0.935310,17,0.0,0.000000,0.000000,0.058824
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,1.0,6151.0,1.0,1490.2,2.659432,9,0.0,0.222222,0.333333,0.333333
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,9.0,8140.0,12.7,1618.0,2.405348,22,0.0,0.000000,0.000000,0.090909
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,6.0,78.0,7.0,64.6,1.269589,3,0.0,0.000000,0.000000,0.333333


Новой сессией считаем действие после паузы более 30 минут. Благодаря этому длинный перерыв между визитами не смешивается с ритмом действий внутри одного визита.

In [457]:
# Каждое событие после паузы более 1800 секунд начинает новую сессию
clean_events['session_no'] = (clean_events.gap_s.isna() | clean_events.gap_s.gt(1800)).groupby(clean_events.cookie_id).cumsum()
sessions = clean_events.groupby(['cookie_id', 'session_no']).agg(
    n_events=('event_ts', 'size'),
    first=('event_ts', 'min'),
    last=('event_ts', 'max'),
)
# Время от первого до последнего события внутри каждой отдельной сессии
sessions['span_s'] = (sessions['last'] - sessions['first']).dt.total_seconds()
# Количество сессий
features['n_sessions'] = sessions.groupby(level=0).size()
# Статистики по событиям в рамках одной сессии
features['max_events_per_session'] = sessions.n_events.groupby(level=0).max()
features['median_events_per_session'] = sessions.n_events.groupby(level=0).median()
features['max_session_span_s'] = sessions.span_s.groupby(level=0).max()

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,gap_cv,n_gaps,gap_le_0_share,gap_le_1_share,gap_le_5_share,gap_le_10_share,n_sessions,max_events_per_session,median_events_per_session,max_session_span_s
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,1.076026,6,0.0,0.166667,0.166667,0.333333,1,7,7.0,167.0
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,2.448360,40,0.0,0.000000,0.075000,0.200000,7,13,4.0,579.0
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,2.212209,35,0.0,0.028571,0.114286,0.171429,7,10,5.0,1957.0
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,2.905849,20,0.0,0.000000,0.000000,0.050000,3,10,8.0,410.0
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,2.471874,27,0.0,0.000000,0.000000,0.000000,4,19,4.0,2632.0
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,0.935310,17,0.0,0.000000,0.000000,0.058824,1,18,18.0,1054.0
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,2.659432,9,0.0,0.222222,0.333333,0.333333,2,6,5.0,476.0
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,2.405348,22,0.0,0.000000,0.000000,0.090909,3,13,7.0,3252.0
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,1.269589,3,0.0,0.000000,0.000000,0.333333,1,4,4.0,95.0


Теперь считаем паузы между событиями внутри одной сессии и статистики по всей куки из всех ее сессий.

In [458]:
within_gap = clean_events.gap_s.where(
    clean_events.gap_s.gt(0) & clean_events.gap_s.le(1800)
)
within_stats = within_gap.groupby(clean_events.cookie_id).agg(['count', 'median', 'mean', 'std'])
features['n_within_gaps'] = within_stats['count']
features['within_gap_median_s'] = within_stats['median']
features['within_gap_mean_s'] = within_stats['mean']
features['within_gap_cv'] = (
    within_stats['std'] / within_stats['mean'].replace(0, np.nan)
)

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,gap_le_5_share,gap_le_10_share,n_sessions,max_events_per_session,median_events_per_session,max_session_span_s,n_within_gaps,within_gap_median_s,within_gap_mean_s,within_gap_cv
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,0.166667,0.333333,1,7,7.0,167.0,6,21.0,27.833333,1.076026
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,0.075000,0.200000,7,13,4.0,579.0,34,50.5,46.617647,0.677747
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,0.114286,0.171429,7,10,5.0,1957.0,29,21.0,83.310345,3.548070
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,0.000000,0.050000,3,10,8.0,410.0,18,52.0,49.500000,0.536076
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,0.000000,0.000000,4,19,4.0,2632.0,24,46.5,132.291667,2.516509
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,0.000000,0.058824,1,18,18.0,1054.0,17,32.0,62.000000,0.935310
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,0.333333,0.333333,2,6,5.0,476.0,8,63.5,87.250000,1.251194
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,0.000000,0.090909,3,13,7.0,3252.0,20,29.5,255.950000,2.163384
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,0.000000,0.333333,1,4,4.0,95.0,3,11.0,31.666667,1.269589


Платформу и User-Agent сводим к простым группам. Сырая строка User-Agent и точные версии браузеров не используем. Наличие курсора оцениваем только среди web-событий.

In [459]:
platform_counts = pd.crosstab(clean_events.cookie_id, clean_events.platform)
platform_counts.head(10)

platform,android,ios,web
cookie_id,,,
ck_00013fdfa0bd37dd,15,0,0
ck_000c95f1408dcb00,0,0,16
ck_000e8c52636e3bec,0,0,34
ck_0010e31baa4a1fb7,0,0,16
ck_0010ec3874fb5378,0,0,74
ck_001722063b94cae0,15,0,0
ck_001e42551cd06e40,0,0,8
ck_0027b5d4107d65d8,15,0,0
ck_0027ba644f9ae2cd,44,0,0


Тут считаем доли платформ, а также создаем семейства User-Agent. Выделим следующие: 'http_client', 'headless', 'avito_app', 'yandex', 'firefox', 'chrome'.

In [460]:
features['platform_main'] = platform_counts.idxmax(axis=1)
for platform in ['web', 'android', 'ios']:
    features[f'platform_{platform}_share'] = (
        platform_counts.get(platform, 0) / features.n_events
    )

ua = cookie_id_groupby.user_agent.first().str.lower()
features['ua_family'] = np.select(
    [ua.str.contains('python|scrapy|curl|go-http-client|node-fetch'),
     ua.str.contains('headless'), ua.str.startswith('avito/'),
     ua.str.contains('yabrowser'), ua.str.contains('firefox'),
     ua.str.contains('chrome')],
    ['http_client', 'headless', 'avito_app', 'yandex', 'firefox', 'chrome'],
    default='other',
)

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,max_session_span_s,n_within_gaps,within_gap_median_s,within_gap_mean_s,within_gap_cv,platform_main,platform_web_share,platform_android_share,platform_ios_share,ua_family
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,167.0,6,21.0,27.833333,1.076026,android,0.0,1.0,0.0,chrome
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,579.0,34,50.5,46.617647,0.677747,web,1.0,0.0,0.0,chrome
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,1957.0,29,21.0,83.310345,3.548070,web,1.0,0.0,0.0,firefox
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,410.0,18,52.0,49.500000,0.536076,web,1.0,0.0,0.0,yandex
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,2632.0,24,46.5,132.291667,2.516509,web,1.0,0.0,0.0,yandex
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,1054.0,17,32.0,62.000000,0.935310,web,1.0,0.0,0.0,chrome
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,476.0,8,63.5,87.250000,1.251194,android,0.0,1.0,0.0,yandex
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,3252.0,20,29.5,255.950000,2.163384,android,0.0,1.0,0.0,chrome
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,95.0,3,11.0,31.666667,1.269589,web,1.0,0.0,0.0,chrome


То же самое сделаем и с операционными системами.

In [461]:
features['os_family'] = np.select(
    [ua.str.contains('android'), ua.str.contains('iphone|ios'),
     ua.str.contains('windows'), ua.str.contains('macintosh'),
     ua.str.contains('linux')],
    ['android', 'ios', 'windows', 'mac', 'linux'],
    default='other',
)

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,n_within_gaps,within_gap_median_s,within_gap_mean_s,within_gap_cv,platform_main,platform_web_share,platform_android_share,platform_ios_share,ua_family,os_family
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,6,21.0,27.833333,1.076026,android,0.0,1.0,0.0,chrome,android
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,34,50.5,46.617647,0.677747,web,1.0,0.0,0.0,chrome,mac
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,29,21.0,83.310345,3.548070,web,1.0,0.0,0.0,firefox,windows
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,18,52.0,49.500000,0.536076,web,1.0,0.0,0.0,yandex,windows
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,24,46.5,132.291667,2.516509,web,1.0,0.0,0.0,yandex,windows
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,17,32.0,62.000000,0.935310,web,1.0,0.0,0.0,chrome,android
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,8,63.5,87.250000,1.251194,android,0.0,1.0,0.0,yandex,windows
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,20,29.5,255.950000,2.163384,android,0.0,1.0,0.0,chrome,android
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,3,11.0,31.666667,1.269589,web,1.0,0.0,0.0,chrome,android


Проверим, была ли смена User-Agent или нет, а также количество смен

In [462]:
features['ua_changes'] = (
    clean_events.user_agent.ne(cookie_id_groupby.user_agent.shift())
    .groupby(clean_events.cookie_id).sum() - 1
)

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,within_gap_median_s,within_gap_mean_s,within_gap_cv,platform_main,platform_web_share,platform_android_share,platform_ios_share,ua_family,os_family,ua_changes
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,21.0,27.833333,1.076026,android,0.0,1.0,0.0,chrome,android,0
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,50.5,46.617647,0.677747,web,1.0,0.0,0.0,chrome,mac,0
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,21.0,83.310345,3.548070,web,1.0,0.0,0.0,firefox,windows,0
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,52.0,49.500000,0.536076,web,1.0,0.0,0.0,yandex,windows,0
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,46.5,132.291667,2.516509,web,1.0,0.0,0.0,yandex,windows,0
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,32.0,62.000000,0.935310,web,1.0,0.0,0.0,chrome,android,0
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,63.5,87.250000,1.251194,android,0.0,1.0,0.0,yandex,windows,0
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,29.5,255.950000,2.163384,android,0.0,1.0,0.0,chrome,android,0
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,11.0,31.666667,1.269589,web,1.0,0.0,0.0,chrome,android,0


In [463]:
features['ua_changes'].value_counts()

ua_changes
0     14847
2        85
3        82
6        80
4        79
      ...  
60        1
56        1
46        1
55        1
49        1
Name: count, Length: 64, dtype: int64

Также отдельно разберем группу web. Посчитаем количество событий, число событий с заполненным pointer_x (то есть с координатой) и их долю среди web-событий.

In [464]:
web_events = clean_events.loc[clean_events.platform == 'web']
features['n_web_events'] = web_events.groupby('cookie_id').size()
features['n_web_pointer'] = web_events.groupby('cookie_id').pointer_x.count()
features['web_pointer_share'] = (features.n_web_pointer / features.n_web_events.replace(0, np.nan))
features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,platform_main,platform_web_share,platform_android_share,platform_ios_share,ua_family,os_family,ua_changes,n_web_events,n_web_pointer,web_pointer_share
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,android,0.0,1.0,0.0,chrome,android,0,NaN,NaN,NaN
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,web,1.0,0.0,0.0,chrome,mac,0,41.0,38.0,0.926829
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,web,1.0,0.0,0.0,firefox,windows,0,36.0,35.0,0.972222
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,web,1.0,0.0,0.0,yandex,windows,0,21.0,0.0,0.000000
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,web,1.0,0.0,0.0,yandex,windows,0,28.0,27.0,0.964286
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,web,1.0,0.0,0.0,chrome,android,0,18.0,16.0,0.888889
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,android,0.0,1.0,0.0,yandex,windows,0,NaN,NaN,NaN
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,android,0.0,1.0,0.0,chrome,android,0,NaN,NaN,NaN
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,web,1.0,0.0,0.0,chrome,android,0,4.0,0.0,0.000000


In [465]:
# Это редкие координаты в событиях, а не полная траектория мыши.
features['pointer_x_std'] = web_events.groupby('cookie_id').pointer_x.std()
features['pointer_y_std'] = web_events.groupby('cookie_id').pointer_y.std()
features['n_unique_pointer_positions'] = (
    web_events[['cookie_id', 'pointer_x', 'pointer_y']]
    .dropna().drop_duplicates().groupby('cookie_id').size()
)
features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,platform_ios_share,ua_family,os_family,ua_changes,n_web_events,n_web_pointer,web_pointer_share,pointer_x_std,pointer_y_std,n_unique_pointer_positions
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,0.0,chrome,android,0,NaN,NaN,NaN,NaN,NaN,NaN
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,0.0,chrome,mac,0,41.0,38.0,0.926829,509.231969,272.953569,38.0
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,0.0,firefox,windows,0,36.0,35.0,0.972222,602.124295,312.027426,35.0
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,0.0,yandex,windows,0,21.0,0.0,0.000000,NaN,NaN,NaN
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,0.0,yandex,windows,0,28.0,27.0,0.964286,450.561957,295.966031,27.0
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,0.0,chrome,android,0,18.0,16.0,0.888889,583.514778,232.205432,16.0
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,0.0,yandex,windows,0,NaN,NaN,NaN,NaN,NaN,NaN
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,0.0,chrome,android,0,NaN,NaN,NaN,NaN,NaN,NaN
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,0.0,chrome,android,0,4.0,0.0,0.000000,NaN,NaN,NaN


Дальше проанализируем поиск. Для поиска важны глубина просмотра выдачи и число поисковых действий. Пропуски `search_page` у других типов событий здесь не считаются признаком бота.

In [466]:
search = clean_events.loc[clean_events.event_name == 'search_results_view']
search_group = search.groupby('cookie_id').search_page
features['search_page_max'] = search_group.max()
features['search_page_median'] = search_group.median()
features['search_page_q90'] = search_group.quantile(0.90)
for page in [3, 5, 10]:
    features[f'search_page_gt_{page}_share'] = (
        search.search_page.gt(page).groupby(search.cookie_id).sum()
        / features.n_search_results_view.replace(0, np.nan)
    )
features['seller_pro_share'] = (
    clean_events.seller_type.eq('pro').groupby(clean_events.cookie_id).sum()
    / features.n_seller_type_present.replace(0, np.nan)
)

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,pointer_x_std,pointer_y_std,n_unique_pointer_positions,search_page_max,search_page_median,search_page_q90,search_page_gt_3_share,search_page_gt_5_share,search_page_gt_10_share,seller_pro_share
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,NaN,NaN,NaN,6.0,2.0,5.2,0.333333,0.333333,0.000000,0.750000
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,509.231969,272.953569,38.0,8.0,3.5,6.6,0.500000,0.250000,0.000000,0.392857
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,602.124295,312.027426,35.0,10.0,2.0,8.6,0.307692,0.230769,0.000000,0.611111
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,NaN,NaN,NaN,2.0,2.0,2.0,0.000000,0.000000,0.000000,0.388889
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,450.561957,295.966031,27.0,7.0,2.0,5.5,0.333333,0.166667,0.000000,0.300000
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,583.514778,232.205432,16.0,16.0,2.0,8.2,0.142857,0.142857,0.142857,0.333333
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,NaN,NaN,NaN,1.0,1.0,1.0,0.000000,0.000000,0.000000,0.444444
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,NaN,NaN,NaN,8.0,3.0,6.0,0.500000,0.166667,0.000000,0.176471
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,NaN,NaN,NaN,6.0,4.5,5.7,0.500000,0.500000,0.000000,1.000000


Обработаем пропуски, бесконечные значений (могли возникнуть при делении на ноль), выделяем категориальные признаки

In [467]:
features = features.replace([np.inf, -np.inf], np.nan)
for column in ['n_viewed_items', 'n_web_events', 'n_web_pointer',
               'n_unique_pointer_positions']:
    features[column] = features[column].fillna(0)
cat_cols = ['platform_main', 'ua_family', 'os_family']
features[cat_cols] = features[cat_cols].fillna('MISSING').astype(str)
features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,pointer_x_std,pointer_y_std,n_unique_pointer_positions,search_page_max,search_page_median,search_page_q90,search_page_gt_3_share,search_page_gt_5_share,search_page_gt_10_share,seller_pro_share
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,NaN,NaN,0.0,6.0,2.0,5.2,0.333333,0.333333,0.000000,0.750000
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,509.231969,272.953569,38.0,8.0,3.5,6.6,0.500000,0.250000,0.000000,0.392857
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,602.124295,312.027426,35.0,10.0,2.0,8.6,0.307692,0.230769,0.000000,0.611111
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,NaN,NaN,0.0,2.0,2.0,2.0,0.000000,0.000000,0.000000,0.388889
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,450.561957,295.966031,27.0,7.0,2.0,5.5,0.333333,0.166667,0.000000,0.300000
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,583.514778,232.205432,16.0,16.0,2.0,8.2,0.142857,0.142857,0.142857,0.333333
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,NaN,NaN,0.0,1.0,1.0,1.0,0.000000,0.000000,0.000000,0.444444
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,NaN,NaN,0.0,8.0,3.0,6.0,0.500000,0.166667,0.000000,0.176471
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,NaN,NaN,0.0,6.0,4.5,5.7,0.500000,0.500000,0.000000,1.000000


In [468]:
assert len(features) == len(meta)
assert features.index.is_unique
assert np.isfinite(features.select_dtypes(include='number').fillna(0).to_numpy()).all()
print(f'Признаков: {features.shape[1]}; куки: {features.shape[0]}')

Признаков: 93; куки: 16000


### Дополнительные признаки

Добавляем характеристики пауз, сессий, концентрации действий и движения указателя.

In [469]:
# Неопределённость оставляем как NaN, число интервалов храним отдельно.
features['gap_q25_s'] = cookie_id_groupby.gap_s.quantile(0.25)
features['gap_q75_s'] = cookie_id_groupby.gap_s.quantile(0.75)
features['gap_iqr_s'] = features.gap_q75_s - features.gap_q25_s
gap_counts = clean_events.dropna(subset=['gap_s']).groupby(['cookie_id', 'gap_s']).size()
features['most_common_gap_share'] = gap_counts.groupby(level=0).max() / features.n_gaps.replace(0, np.nan)

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,search_page_median,search_page_q90,search_page_gt_3_share,search_page_gt_5_share,search_page_gt_10_share,seller_pro_share,gap_q25_s,gap_q75_s,gap_iqr_s,most_common_gap_share
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,2.0,5.2,0.333333,0.333333,0.000000,0.750000,9.50,31.00,21.50,0.166667
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,3.5,6.6,0.500000,0.250000,0.000000,0.392857,19.00,82.25,63.25,0.050000
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,2.0,8.6,0.307692,0.230769,0.000000,0.611111,14.50,73.00,58.50,0.085714
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,2.0,2.0,0.000000,0.000000,0.000000,0.388889,37.25,70.00,32.75,0.100000
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,2.0,5.5,0.333333,0.166667,0.000000,0.300000,32.50,106.50,74.00,0.074074
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,2.0,8.2,0.142857,0.142857,0.142857,0.333333,23.00,114.00,91.00,0.117647
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,1.0,1.0,0.000000,0.000000,0.000000,0.444444,2.00,141.00,139.00,0.222222
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,3.0,6.0,0.500000,0.166667,0.000000,0.176471,21.00,69.50,48.50,0.090909
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,4.5,5.7,0.500000,0.500000,0.000000,1.000000,8.50,44.50,36.00,0.333333


In [470]:
# Энтропия пауз по заранее заданным диапазонам.
gap_bins = pd.cut(clean_events.gap_s, [-1, 0, 1, 5, 10, 30, 60, 300, 1800, np.inf])
gap_bin_counts = clean_events.groupby(['cookie_id', gap_bins], observed=True).size()
gap_bin_prob = gap_bin_counts / gap_bin_counts.groupby(level=0).transform('sum')
features['gap_bin_entropy'] = -(gap_bin_prob * np.log(gap_bin_prob)).groupby(level=0).sum()

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,search_page_q90,search_page_gt_3_share,search_page_gt_5_share,search_page_gt_10_share,seller_pro_share,gap_q25_s,gap_q75_s,gap_iqr_s,most_common_gap_share,gap_bin_entropy
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,5.2,0.333333,0.333333,0.000000,0.750000,9.50,31.00,21.50,0.166667,1.560710
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,6.6,0.500000,0.250000,0.000000,0.392857,19.00,82.25,63.25,0.050000,1.685863
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,8.6,0.307692,0.230769,0.000000,0.611111,14.50,73.00,58.50,0.085714,1.701221
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,2.0,0.000000,0.000000,0.000000,0.388889,37.25,70.00,32.75,0.100000,1.430562
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,5.5,0.333333,0.166667,0.000000,0.300000,32.50,106.50,74.00,0.074074,1.410908
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,8.2,0.142857,0.142857,0.142857,0.333333,23.00,114.00,91.00,0.117647,1.242255
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,1.0,0.000000,0.000000,0.000000,0.444444,2.00,141.00,139.00,0.222222,1.676988
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,6.0,0.500000,0.166667,0.000000,0.176471,21.00,69.50,48.50,0.090909,1.570383
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,5.7,0.500000,0.500000,0.000000,1.000000,8.50,44.50,36.00,0.333333,1.098612


In [471]:
# Длина и плотность сессий, а также ночная активность по времени датасета.
features['median_session_span_s'] = sessions.span_s.groupby(level=0).median()
features['mean_events_per_session'] = sessions.n_events.groupby(level=0).mean()
features['singleton_session_share'] = sessions.n_events.eq(1).groupby(level=0).sum() / features.n_sessions
features['night_event_share'] = clean_events.event_ts.dt.hour.lt(6).groupby(clean_events.cookie_id).sum() / features.n_events
features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,seller_pro_share,gap_q25_s,gap_q75_s,gap_iqr_s,most_common_gap_share,gap_bin_entropy,median_session_span_s,mean_events_per_session,singleton_session_share,night_event_share
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,0.750000,9.50,31.00,21.50,0.166667,1.560710,167.0,7.000000,0.00,1.00000
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,0.392857,19.00,82.25,63.25,0.050000,1.685863,153.0,5.857143,0.00,0.00000
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,0.611111,14.50,73.00,58.50,0.085714,1.701221,70.0,5.142857,0.00,0.50000
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,0.388889,37.25,70.00,32.75,0.100000,1.430562,392.0,7.000000,0.00,0.52381
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,0.300000,32.50,106.50,74.00,0.074074,1.410908,271.5,7.000000,0.25,0.00000
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,0.333333,23.00,114.00,91.00,0.117647,1.242255,1054.0,18.000000,0.00,0.00000
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,0.444444,2.00,141.00,139.00,0.222222,1.676988,349.0,5.000000,0.00,1.00000
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,0.176471,21.00,69.50,48.50,0.090909,1.570383,1775.0,7.666667,0.00,0.00000
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,1.000000,8.50,44.50,36.00,0.333333,1.098612,95.0,4.000000,0.00,0.00000


Концентрацию по категориям, локациям и запросам считаем только по непустым значениям.

In [472]:
for column in ['item_category', 'item_location', 'search_query']:
    counts = clean_events.dropna(subset=[column]).groupby(['cookie_id', column]).size()
    probability = counts / counts.groupby(level=0).transform('sum')
    features[f'{column}_top_share'] = counts.groupby(level=0).max() / features[f'n_{column}_present'].replace(0, np.nan)
    features[f'{column}_entropy'] = -(probability * np.log(probability)).groupby(level=0).sum()

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,median_session_span_s,mean_events_per_session,singleton_session_share,night_event_share,item_category_top_share,item_category_entropy,item_location_top_share,item_location_entropy,search_query_top_share,search_query_entropy
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,167.0,7.000000,0.00,1.00000,0.666667,0.867563,0.428571,1.475076,0.333333,1.098612
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,153.0,5.857143,0.00,0.00000,1.000000,-0.000000,0.324324,2.085509,0.250000,1.559581
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,70.0,5.142857,0.00,0.50000,0.266667,1.832968,0.666667,1.293930,0.230769,2.204785
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,392.0,7.000000,0.00,0.52381,1.000000,-0.000000,0.500000,1.237597,0.333333,1.098612
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,271.5,7.000000,0.25,0.00000,0.400000,1.278780,0.740741,0.954708,0.166667,1.791759
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,1054.0,18.000000,0.00,0.00000,0.294118,1.498740,0.666667,1.079735,0.285714,1.747868
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,349.0,5.000000,0.00,1.00000,0.428571,1.078992,0.300000,1.504788,1.000000,-0.000000
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,1775.0,7.666667,0.00,0.00000,1.000000,-0.000000,0.217391,1.941785,0.666667,0.867563
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,95.0,4.000000,0.00,0.00000,0.666667,0.636514,0.500000,1.039721,0.500000,0.693147


Смены User-Agent считаем между разными секундами. Секунды с несколькими значениями исключаем, так как порядок событий внутри секунды неизвестен.

In [473]:
ua_seconds = clean_events.groupby(['cookie_id', 'event_ts']).user_agent.agg(['first', 'nunique']).reset_index()
features['ua_conflicted_seconds'] = (ua_seconds['nunique'].gt(1).groupby(ua_seconds.cookie_id).sum())
ua_seconds = ua_seconds.loc[ua_seconds['nunique'].eq(1)].copy()
features['ua_changes'] = (
    ua_seconds['first'].ne(ua_seconds.groupby('cookie_id')['first'].shift())
    .groupby(ua_seconds.cookie_id).sum() - 1
)
features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,mean_events_per_session,singleton_session_share,night_event_share,item_category_top_share,item_category_entropy,item_location_top_share,item_location_entropy,search_query_top_share,search_query_entropy,ua_conflicted_seconds
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,7.000000,0.00,1.00000,0.666667,0.867563,0.428571,1.475076,0.333333,1.098612,0
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,5.857143,0.00,0.00000,1.000000,-0.000000,0.324324,2.085509,0.250000,1.559581,0
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,5.142857,0.00,0.50000,0.266667,1.832968,0.666667,1.293930,0.230769,2.204785,0
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,7.000000,0.00,0.52381,1.000000,-0.000000,0.500000,1.237597,0.333333,1.098612,0
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,7.000000,0.25,0.00000,0.400000,1.278780,0.740741,0.954708,0.166667,1.791759,0
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,18.000000,0.00,0.00000,0.294118,1.498740,0.666667,1.079735,0.285714,1.747868,0
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,5.000000,0.00,1.00000,0.428571,1.078992,0.300000,1.504788,1.000000,-0.000000,0
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,7.666667,0.00,0.00000,1.000000,-0.000000,0.217391,1.941785,0.666667,0.867563,0
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,4.000000,0.00,0.00000,0.666667,0.636514,0.500000,1.039721,0.500000,0.693147,0


Между доступными точками указателя в разные секунды измеряем расстояние. Это не непрерывная траектория мыши.

In [474]:
pointer = web_events[['cookie_id', 'event_ts', 'pointer_x', 'pointer_y']].dropna().copy()
pointer_groups = pointer.groupby(['cookie_id', 'event_ts'])
ambiguous = pointer_groups[['pointer_x', 'pointer_y']].nunique().max(axis=1).gt(1)
pointer = pointer_groups.first().loc[~ambiguous].reset_index()
previous_point = pointer.groupby('cookie_id')[['pointer_x', 'pointer_y']].shift()
pointer['move_px'] = np.hypot(
    pointer.pointer_x - previous_point.pointer_x,
    pointer.pointer_y - previous_point.pointer_y,
)
pointer_moves = pointer.groupby('cookie_id').move_px
features['n_pointer_moves'] = pointer_moves.count()
features['pointer_move_median_px'] = pointer_moves.median()
features['pointer_move_q90_px'] = pointer_moves.quantile(0.90)

features.head(10)

,n_events,n_contact_chat_open,share_contact_chat_open,n_contact_message_sent,share_contact_message_sent,n_contact_phone_show,share_contact_phone_show,n_favorite_add,share_favorite_add,n_item_view,...,item_category_top_share,item_category_entropy,item_location_top_share,item_location_entropy,search_query_top_share,search_query_entropy,ua_conflicted_seconds,n_pointer_moves,pointer_move_median_px,pointer_move_q90_px
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,7,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,...,0.666667,0.867563,0.428571,1.475076,0.333333,1.098612,0,NaN,NaN,NaN
ck_7e4de46eeab82974,41,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,...,1.000000,-0.000000,0.324324,2.085509,0.250000,1.559581,0,37.0,659.794665,1132.155401
ck_9320229ef6304522,36,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,...,0.266667,1.832968,0.666667,1.293930,0.230769,2.204785,0,34.0,855.446249,1592.977349
ck_30ccd25bc1714ed9,21,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,...,1.000000,-0.000000,0.500000,1.237597,0.333333,1.098612,0,NaN,NaN,NaN
ck_a77c5f05948cdeef,28,0,0.000000,0,0.000000,1,0.035714,2,0.071429,12,...,0.400000,1.278780,0.740741,0.954708,0.166667,1.791759,0,26.0,599.205444,1079.379611
ck_ea6f843af2292fec,18,0,0.000000,1,0.055556,1,0.055556,3,0.166667,4,...,0.294118,1.498740,0.666667,1.079735,0.285714,1.747868,0,15.0,772.187153,1219.262988
ck_e38634dc2f188574,10,1,0.100000,0,0.000000,1,0.100000,0,0.000000,3,...,0.428571,1.078992,0.300000,1.504788,1.000000,-0.000000,0,NaN,NaN,NaN
ck_ad7f01440bb250d1,23,1,0.043478,2,0.086957,0,0.000000,3,0.130435,9,...,1.000000,-0.000000,0.217391,1.941785,0.666667,0.867563,0,NaN,NaN,NaN
ck_a7a58d28a4482bdc,4,0,0.000000,0,0.000000,0,0.000000,1,0.250000,1,...,0.666667,0.636514,0.500000,1.039721,0.500000,0.693147,0,NaN,NaN,NaN


In [475]:
# Нулём обозначаем отсутствие наблюдений только у счётчиков.
features = features.replace([np.inf, -np.inf], np.nan)
constant_cols = features.columns[features.nunique(dropna=False).eq(1)].tolist()
features = features.drop(columns=constant_cols).copy()
for column in ['n_pointer_moves', 'ua_changes', 'ua_conflicted_seconds']:
    features[column] = features[column].fillna(0)
assert features.index.equals(pd.Index(meta.cookie_id))
assert np.isfinite(features.select_dtypes(include='number').fillna(0).to_numpy()).all()
print(f'Признаков: {features.shape[1]}; удалены константы: {constant_cols}')

Признаков: 110; удалены константы: ['created_in_window', 'n_platform_unique']


### Вывод по этапу №3

Для каждой куки собрали одну строку из 110 признаков: состав и частоту действий, разнообразие объявлений и запросов, возраст куки, время активности, паузы и сессии, платформу и User-Agent, глубину поиска и характеристики доступных координат указателя. Признаки рассчитаны только по событиям внутри окна наблюдения, без использования target.

## 4. Первая модель CatBoost

Сравниваем CatBoost с quickstart baseline на одних датах: обучение до 13 апреля, проверка 13–16 апреля. Основная метрика берётся из metric.py.

In [476]:
X_train = features.loc[train.cookie_id].reset_index(drop=True)
X_test = features.loc[test.cookie_id].reset_index(drop=True)
y = train.target.reset_index(drop=True)
valid_from = pd.Timestamp('2026-04-13')
valid_to = pd.Timestamp('2026-04-17')
fit_rows = train.window_start_ts.lt(valid_from).to_numpy()
valid_rows = train.window_start_ts.between(valid_from, valid_to, inclusive='left').to_numpy()
y_B = y.loc[valid_rows]
print(f'Train: {fit_rows.sum()}, valid: {valid_rows.sum()}, bots: {y_B.sum()}')

Train: 5930, valid: 3210, bots: 248


In [477]:
# Baseline на тех же датах B.
baseline_B = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=3, random_state=SEED
)
baseline_B.fit(X_baseline.loc[fit_rows, feature_cols], y.loc[fit_rows])
baseline_score_B = baseline_B.predict_proba(X_baseline.loc[valid_rows, feature_cols])[:, 1]

In [478]:
model_B = CatBoostClassifier(
    iterations=1200, depth=5, learning_rate=0.05, l2_leaf_reg=10,
    loss_function='Logloss', random_seed=SEED, thread_count=4,
    allow_writing_files=False, verbose=False,
)
model_B.fit(
    X_train.loc[fit_rows], y.loc[fit_rows], cat_features=cat_cols,
    eval_set=(X_train.loc[valid_rows], y_B), early_stopping_rounds=150,
)
catboost_score_B = model_B.predict_proba(X_train.loc[valid_rows])[:, 1]
print(f'Лучшее число деревьев: {model_B.best_iteration_ + 1}')

Лучшее число деревьев: 675


In [479]:
comparison_B = pd.DataFrame({
    'model': ['quickstart_baseline_B', 'catboost_v1_B'],
    'precision_at_recall_70': [
        precision_at_recall(y_B, baseline_score_B),
        precision_at_recall(y_B, catboost_score_B),
    ],
    'average_precision': [
        average_precision_score(y_B, baseline_score_B),
        average_precision_score(y_B, catboost_score_B),
    ],
    'roc_auc': [
        roc_auc_score(y_B, baseline_score_B),
        roc_auc_score(y_B, catboost_score_B),
    ],
})
display(comparison_B)

,model,precision_at_recall_70,average_precision,roc_auc
0,quickstart_baseline_B,0.096703,0.236297,0.631622
1,catboost_v1_B,0.685039,0.773504,0.939263


Давайте посмотрим на важность признаков обученной модели.

In [480]:
importance_B = pd.Series(model_B.feature_importances_, index=X_train.columns).sort_values(ascending=False)
display(importance_B.head(20).to_frame('importance'))

,importance
pointer_x_std,6.470797
unique_item_id_share,5.227435
pointer_move_q90_px,3.933367
within_gap_median_s,3.019929
gap_q75_s,2.903318
within_gap_cv,2.763908
n_item_category_unique,2.680419
unique_item_category_share,2.452434
gap_median_s,2.433131
item_category_entropy,2.110794


Среди первых важных - разброс pointer_x, разнообразие объявлений, движения указателя и паузы. Но сами по себе эти признаки не доказывают, что каждый из них улучшает результат.

Признаки и оценки на B сохраняем локально для следующих экспериментов.

In [481]:
Path('artifacts').mkdir(exist_ok=True)
features.to_pickle('artifacts/features_v1.pkl')

In [482]:
predictions_B = train.loc[valid_rows, ['cookie_id', 'target']].copy()
predictions_B['baseline_score'] = baseline_score_B
predictions_B['catboost_score'] = catboost_score_B
predictions_B.to_csv('artifacts/validation_B_v1.csv', index=False)

In [483]:
experiment_rows = []
for _, result in comparison_B.iterrows():
    experiment_rows.append({
        'experiment': result['model'],
        'train_before': str(valid_from.date()),
        'valid_from': str(valid_from.date()),
        'n_train': int(fit_rows.sum()),
        'n_valid': int(valid_rows.sum()),
        'n_valid_positive': int(y_B.sum()),
        'precision_at_recall_70': result['precision_at_recall_70'],
        'average_precision': result['average_precision'],
        'roc_auc': result['roc_auc'],
        'random_state': SEED,
        'deduplicate': result['model'] == 'catboost_v1_B',
        'n_features': X_train.shape[1] if result['model'] == 'catboost_v1_B' else 2,
        'best_iteration': model_B.best_iteration_ + 1 if result['model'] == 'catboost_v1_B' else np.nan,
    })

In [484]:
previous = pd.read_csv(report_path) if report_path.exists() else pd.DataFrame()
names = [row['experiment'] for row in experiment_rows]
if not previous.empty:
    previous = previous.loc[~previous.experiment.isin(names)]
pd.concat([previous, pd.DataFrame(experiment_rows)], ignore_index=True).to_csv(report_path, index=False)

### Вывод по этапу №4.

На разбиении B CatBoost получил **Precision при Recall ≥ 70% = 0.6850** против 0.0967 у quickstart baseline. Дополнительные метрики также выросли: **Average Precision - 0.7735** против 0.2363, **ROC-AUC** - 0.9393 против 0.6316. Устойчивость ещё предстоит проверить на других временных разбиениях.

## 5. Первый кандидат на отправку

После выбора числа деревьев на B обучаем модель на всём train и сохраняем оценки test.

In [485]:
final_v1 = CatBoostClassifier(
    iterations=model_B.best_iteration_ + 1,
    depth=5, learning_rate=0.05, l2_leaf_reg=10,
    loss_function='Logloss', random_seed=SEED, thread_count=4,
    allow_writing_files=False, verbose=False,
)
final_v1.fit(X_train, y, cat_features=cat_cols)

CatBoostClassifier(allow_writing_files=False, depth=5, iterations=675, l2_leaf_reg=10, learning_rate=0.05, loss_function='Logloss', random_seed=42, thread_count=4, verbose=False)

In [486]:
# Привязываем предсказания к cookie_id и сохраняем порядок шаблона.
assert test.cookie_id.is_unique and sample_submission.cookie_id.is_unique
assert set(test.cookie_id) == set(sample_submission.cookie_id)

In [487]:
test_scores = pd.Series(final_v1.predict_proba(X_test)[:, 1], index=test.cookie_id)
submission_v1 = sample_submission[['cookie_id']].copy()
submission_v1['score'] = submission_v1.cookie_id.map(test_scores)

In [488]:
assert submission_v1.columns.tolist() == sample_submission.columns.tolist()
assert submission_v1.cookie_id.equals(sample_submission.cookie_id)
assert np.isfinite(submission_v1.score).all()
assert submission_v1.score.between(0, 1).all()

In [489]:
submission_v1.to_csv('artifacts/submission_v1.csv', index=False)
print(f'Сохранено {len(submission_v1)} строк в artifacts/submission_v1.csv')

Сохранено 4909 строк в artifacts/submission_v1.csv


## 6. Отбор признаков

На этапе экспериментов мы проверили удаление восьми групп признаков и добавление новых признаков последовательности, поиска, интересов и ритма. Подробные результаты сохранены локально в reports/experiments.csv. Полный набор v2 (138 признаков) улучшил B, но не подтвердил улучшение на A, поэтому его генерацию в итоговом ноутбуке не оставляем.

| Набор | Признаков | B: P@R≥0.70 | A: P@R≥0.70 |
|---|---:|---:|---:|
| v1 | 110 | 0.6850 | 0.7156 |
| v1 без UA/platform | 97 | 0.7028 | 0.7207 |
| полный v2 | 138 | 0.7085 | 0.7111 |

Для дальнейшей настройки берём компактный набор без UA/platform. Прирост основной метрики небольшой, а AP и ROC-AUC слегка снизились, поэтому v1 сохраняем как резерв.

In [523]:
# Убираем только техническую группу, которую проверили на A и B.
technical_prefixes = (
    'ua_', 'n_user_agent', 'unique_user_agent', 'os_family',
    'platform_', 'n_platform', 'unique_platform',
)
technical_cols = [
    column for column in features.columns
    if column.startswith(technical_prefixes)
]
selected_cols = [
    column for column in features.columns
    if column not in technical_cols
]
assert len(features.columns) == 110 and len(selected_cols) == 97
print(f'Исключено {len(technical_cols)} признаков; осталось {len(selected_cols)}')

Исключено 13 признаков; осталось 97


Сравниваем v1 и сокращённый набор на двух временных разбиениях. Число деревьев и остальные параметры одинаковые, чтобы оценить именно состав признаков.

In [524]:
stage3_iterations = model_B.best_iteration_ + 1
fit_A = train.window_start_ts.lt('2026-04-10').to_numpy()
valid_A = train.window_start_ts.between(
    '2026-04-10', '2026-04-13', inclusive='left'
).to_numpy()
stage3_splits = {
    'B': (fit_rows, valid_rows),
    'A': (fit_A, valid_A),
}
feature_sets = {
    'v1': features.columns.tolist(),
    'v1_no_ua': selected_cols,
}

In [525]:
def predict_on_split(columns, train_rows, validation_rows):
    model = CatBoostClassifier(
        iterations=stage3_iterations, depth=5, learning_rate=0.05,
        l2_leaf_reg=10, loss_function='Logloss', random_seed=SEED,
        thread_count=4, allow_writing_files=False, verbose=False,
    )
    model.fit(
        X_train.loc[train_rows, columns], y.loc[train_rows],
        cat_features=[column for column in cat_cols if column in columns],
    )
    return model.predict_proba(X_train.loc[validation_rows, columns])[:, 1]

In [526]:
stage3_rows = []
for split_name, (train_rows, validation_rows) in stage3_splits.items():
    target = y.loc[validation_rows]
    for name, columns in feature_sets.items():
        scores = predict_on_split(columns, train_rows, validation_rows)
        stage3_rows.append({
            'experiment': f'{name}_{split_name}',
            'precision_at_recall_70': precision_at_recall(target, scores),
            'average_precision': average_precision_score(target, scores),
            'roc_auc': roc_auc_score(target, scores),
            'n_features': len(columns),
        })
stage3_results = pd.DataFrame(stage3_rows)
display(stage3_results)

,experiment,precision_at_recall_70,average_precision,roc_auc,n_features
0,v1_B,0.685039,0.773504,0.939263,110
1,v1_no_ua_B,0.702811,0.763550,0.934018,97
2,v1_A,0.715556,0.757165,0.916147,110
3,v1_no_ua_A,0.720721,0.754164,0.914842,97


In [527]:
# Журнал обновляем без дублирования строк при повторном запуске.
stage3_results['random_state'] = SEED
stage3_results['iterations'] = stage3_iterations
stage3_results['train_before'] = np.where(
    stage3_results.experiment.str.endswith('_A'), '2026-04-10', '2026-04-13'
)
stage3_results['valid_from'] = stage3_results.train_before
previous = pd.read_csv(report_path) if report_path.exists() else pd.DataFrame()
if not previous.empty:
    previous = previous.loc[~previous.experiment.isin(stage3_results.experiment)]
pd.concat([previous, stage3_results], ignore_index=True).to_csv(
    report_path, index=False
)

Обучаем сокращённый вариант на всём размеченном train и сохраняем отдельный кандидат.

In [528]:
stage3_model = CatBoostClassifier(
    iterations=stage3_iterations, depth=5, learning_rate=0.05,
    l2_leaf_reg=10, loss_function='Logloss', random_seed=SEED,
    thread_count=4, allow_writing_files=False, verbose=False,
)
stage3_model.fit(X_train[selected_cols], y)

CatBoostClassifier(allow_writing_files=False, depth=5, iterations=675, l2_leaf_reg=10, learning_rate=0.05, loss_function='Logloss', random_seed=42, thread_count=4, verbose=False)

In [529]:
score_by_cookie = pd.Series(
    stage3_model.predict_proba(X_test[selected_cols])[:, 1],
    index=test.cookie_id,
)
submission_stage3 = sample_submission[['cookie_id']].copy()
submission_stage3['score'] = submission_stage3.cookie_id.map(score_by_cookie)
assert submission_stage3.cookie_id.equals(sample_submission.cookie_id)
assert np.isfinite(submission_stage3.score).all()
assert submission_stage3.score.between(0, 1).all()
submission_stage3.to_csv('artifacts/submission_stage3_no_ua.csv', index=False)
print(f'Сохранено {len(submission_stage3)} строк: artifacts/submission_stage3_no_ua.csv')

Сохранено 4909 строк: artifacts/submission_stage3_no_ua.csv


### Вывод по этапу

Дополнительные признаки увеличили набор со 110 до 138, но улучшение на B (0.6850 -> 0.7085) не подтвердилось на A (0.7156 -> 0.7111). Удаление группы UA/platform сократило набор до 97 признаков и немного повысило основную метрику на обоих разбиениях: до **0.7028 на B** и **0.7207 на A**. При этом AP и ROC-AUC слегка снизились, поэтому сокращённый набор оставляем кандидатом для настройки модели, а исходный v1 - резервом.

## 7. Настройка CatBoost

Проверяем пять заранее выбранных конфигураций на двух наборах признаков. На B подбираем параметры и число деревьев по редкой сетке; лучшие варианты перепроверяем на A. Контроль C пока не используем.

In [ ]:
tuning_sets = {
    'v1': features.columns.tolist(),
    'no_ua': selected_cols,
}
tuning_configs = [
    {'depth': 4, 'learning_rate': 0.05, 'l2_leaf_reg': 10},
    {'depth': 5, 'learning_rate': 0.05, 'l2_leaf_reg': 10},
    {'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 10},
    {'depth': 7, 'learning_rate': 0.03, 'l2_leaf_reg': 30},
    {'depth': 5, 'learning_rate': 0.08, 'l2_leaf_reg': 3},
]

print(f'Обучений на B: {len(tuning_sets) * len(tuning_configs)}')

Обучений на B: 10


In [531]:
def fit_on_B(columns, params):
    model = CatBoostClassifier(
        iterations=1600, loss_function='Logloss', random_seed=SEED,
        thread_count=4, allow_writing_files=False, verbose=False,
        **params,
    )
    model.fit(
        X_train.loc[fit_rows, columns], y.loc[fit_rows],
        cat_features=[c for c in cat_cols if c in columns],
        eval_set=(X_train.loc[valid_rows, columns], y_B),
        early_stopping_rounds=200, use_best_model=False,
    )
    return model

Для каждой обученной модели измеряем официальную метрику и Average Precision примерно через каждые 100 деревьев. Поскольку это грубая сетка: точный максимум после каждого дерева был бы слишком шумным на 248 ботах B.

In [532]:
checkpoint_rows = []
checkpoint_scores_B = {}

for set_name, columns in tuning_sets.items():
    for config_id, params in enumerate(tuning_configs):
        started = perf_counter()
        model = fit_on_B(columns, params)

        checkpoints = list(range(100, model.tree_count_ + 1, 100))
        checkpoints += [model.best_iteration_ + 1, model.tree_count_]

        for trees in sorted(set(checkpoints)):
            scores = model.predict_proba(X_train.loc[valid_rows, columns], ntree_end=trees)[:, 1]
            key = (set_name, config_id, trees)

            checkpoint_scores_B[key] = scores
            checkpoint_rows.append({
                'feature_set': set_name, 'config_id': config_id,
                'trees': trees,
                'precision_at_recall_70': precision_at_recall(y_B, scores),
                'average_precision': average_precision_score(y_B, scores),
                'logloss_best_tree': model.best_iteration_ + 1,
                'fit_seconds': round(perf_counter() - started, 1),
            })
            
        print(set_name, config_id, 'trees:', model.tree_count_)

v1 0 trees: 968
v1 1 trees: 875
v1 2 trees: 742
v1 3 trees: 948
v1 4 trees: 457
no_ua 0 trees: 859
no_ua 1 trees: 631
no_ua 2 trees: 572
no_ua 3 trees: 1576
no_ua 4 trees: 440


In [533]:
checkpoint_B = pd.DataFrame(checkpoint_rows)
best_by_config_B = (checkpoint_B.sort_values(['precision_at_recall_70', 'average_precision'], ascending=False)
    .drop_duplicates(['feature_set', 'config_id'])
    .sort_values('precision_at_recall_70', ascending=False)
)

display(best_by_config_B)
checkpoint_B.to_csv('artifacts/stage4_checkpoints_B.csv', index=False)

,feature_set,config_id,trees,precision_at_recall_70,average_precision,logloss_best_tree,fit_seconds
61,no_ua,1,431,0.701613,0.764216,431,4.2
43,v1,4,257,0.698795,0.769620,257,23.8
23,v1,2,300,0.694444,0.768326,542,46.2
81,no_ua,3,1000,0.693227,0.770047,1376,29.9
19,v1,1,800,0.690476,0.773319,675,46.3
34,v1,3,500,0.687747,0.767761,748,75.0
9,v1,0,900,0.687747,0.765796,768,42.7
90,no_ua,4,200,0.686275,0.758997,240,3.2
50,no_ua,0,400,0.674419,0.762977,659,3.9
67,no_ua,2,300,0.661654,0.762782,372,6.0


На A проверяем по два лучших варианта каждого набора. Число деревьев уже выбрано на B и здесь не меняется.

In [534]:
finalists_B = (best_by_config_B.sort_values(['feature_set', 'precision_at_recall_70'], ascending=[True, False])
    .groupby('feature_set').head(2)
    .reset_index(drop=True)
)

display(finalists_B[['feature_set', 'config_id', 'trees',
                     'precision_at_recall_70', 'average_precision']])

,feature_set,config_id,trees,precision_at_recall_70,average_precision
0,no_ua,1,431,0.701613,0.764216
1,no_ua,3,1000,0.693227,0.770047
2,v1,4,257,0.698795,0.769620
3,v1,2,300,0.694444,0.768326


In [535]:
finalist_rows_A = []
finalist_scores_A = {}
finalist_scores_B = {}

for row in finalists_B.itertuples():
    columns = tuning_sets[row.feature_set]
    params = tuning_configs[row.config_id]

    model = CatBoostClassifier(
        iterations=int(row.trees), loss_function='Logloss',
        random_seed=SEED, thread_count=4,
        allow_writing_files=False, verbose=False, **params,
    )

    model.fit(
        X_train.loc[fit_A, columns], y.loc[fit_A],
        cat_features=[c for c in cat_cols if c in columns],
    )

    key = (row.feature_set, row.config_id)
    scores_A = model.predict_proba(X_train.loc[valid_A, columns])[:, 1]

    finalist_scores_A[key] = scores_A
    finalist_scores_B[key] = checkpoint_scores_B[(row.feature_set, row.config_id, row.trees)]

    finalist_rows_A.append({
        'feature_set': row.feature_set, 'config_id': row.config_id,
        'trees': row.trees,
        'precision_A': precision_at_recall(y.loc[valid_A], scores_A),
        'average_precision_A': average_precision_score(y.loc[valid_A], scores_A),
        'precision_B': row.precision_at_recall_70,
        'average_precision_B': row.average_precision,
    })

finalists_AB = pd.DataFrame(finalist_rows_A)

display(finalists_AB.sort_values(['precision_B', 'precision_A'], ascending=False))

,feature_set,config_id,trees,precision_A,average_precision_A,precision_B,average_precision_B
0,no_ua,1,431,0.723982,0.754569,0.701613,0.764216
2,v1,4,257,0.686695,0.748059,0.698795,0.769620
3,v1,2,300,0.735160,0.763832,0.694444,0.768326
1,no_ua,3,1000,0.774038,0.769398,0.693227,0.770047


### Веса класса

Проверяем веса ботов 2 и 4 для перспективного CatBoost без UA/platform. Сравниваем результаты на тех же временных проверках B и A.

In [537]:
weight_scores = {
    1: {
        'B': finalist_scores_B[('no_ua', 3)],
        'A': finalist_scores_A[('no_ua', 3)],
    }
}

weight_params = tuning_configs[3]

weight_trees = int(finalists_B.loc[finalists_B.feature_set.eq('no_ua') & finalists_B.config_id.eq(3), 'trees'].iloc[0])

print('Опорная модель:', weight_params, 'деревьев:', weight_trees)

Опорная модель: {'depth': 7, 'learning_rate': 0.03, 'l2_leaf_reg': 30} деревьев: 1000


In [538]:
for bot_weight in [2, 4]:
    weight_scores[bot_weight] = {}

    for split_name, (train_rows, validation_rows) in stage3_splits.items():
        model = CatBoostClassifier(
            iterations=weight_trees, loss_function='Logloss',
            class_weights=[1, bot_weight], random_seed=SEED,
            thread_count=4, allow_writing_files=False, verbose=False,
            **weight_params,
        )

        model.fit(X_train.loc[train_rows, selected_cols], y.loc[train_rows])

        weight_scores[bot_weight][split_name] = model.predict_proba(X_train.loc[validation_rows, selected_cols])[:, 1]
    
    print('Вес бота:', bot_weight)

Вес бота: 2
Вес бота: 4


In [539]:
weight_rows = []
for bot_weight, split_scores in weight_scores.items():
    for split_name, scores in split_scores.items():
        target = y.loc[stage3_splits[split_name][1]]

        weight_rows.append({
            'bot_weight': bot_weight, 'split': split_name,
            'precision_at_recall_70': precision_at_recall(target, scores),
            'average_precision': average_precision_score(target, scores),
        })

weight_results = pd.DataFrame(weight_rows)

display(weight_results)

,bot_weight,split,precision_at_recall_70,average_precision
0,1,B,0.693227,0.770047
1,1,A,0.774038,0.769398
2,2,B,0.719008,0.770543
3,2,A,0.769231,0.766445
4,4,B,0.713115,0.767777
5,4,A,0.805000,0.771672


In [544]:
# Сохраняем прогнозы двух финалистов на A и B для контроля и сравнения.
for split_name, (_, validation_rows) in stage3_splits.items():
    predictions = train.loc[validation_rows, ['cookie_id', 'target']].copy()

    for bot_weight in [2, 4]:
        predictions[f'weight_{bot_weight}'] = weight_scores[bot_weight][split_name]
        
    predictions.to_csv(f'artifacts/stage4_weighted_{split_name}.csv', index=False)

In [545]:
weight_log = weight_results.copy()

weight_log['experiment'] = ('stage4_weight_' + weight_log.bot_weight.astype(str) + '_' + weight_log.split)
weight_log['depth'] = weight_params['depth']
weight_log['learning_rate'] = weight_params['learning_rate']
weight_log['l2_leaf_reg'] = weight_params['l2_leaf_reg']
weight_log['iterations'] = weight_trees
weight_log['n_features'] = len(selected_cols)
weight_log['random_state'] = SEED

previous = pd.read_csv(report_path)
previous = previous.loc[~previous.experiment.isin(weight_log.experiment)]

pd.concat([previous, weight_log], ignore_index=True).to_csv(report_path, index=False)

### Выбор для контрольного периода C

Фиксируем два рецепта CatBoost на 97 признаках: depth=7, learning_rate=0.03, l2_leaf_reg=30, 1000 деревьев, seed=42; веса бота 4 и 2. Вес 4 дал P@R≥0.70 0.713 на B и 0.805 на A; вес 2 — 0.719 и 0.769. Отправленный v1 остаётся резервом с известным скрытым результатом 0.78589. Следующий шаг — один контроль на C без изменения этих рецептов.

In [546]:
stage4_recipes = {
    f'weighted_{bot_weight}': {
        'columns': selected_cols,
        'iterations': weight_trees,
        'depth': weight_params['depth'],
        'learning_rate': weight_params['learning_rate'],
        'l2_leaf_reg': weight_params['l2_leaf_reg'],
        'class_weights': [1, bot_weight],
    }
    for bot_weight in [4, 2]
}

print({name: {k: v for k, v in recipe.items() if k != 'columns'}
       for name, recipe in stage4_recipes.items()})

{'weighted_4': {'iterations': 1000, 'depth': 7, 'learning_rate': 0.03, 'l2_leaf_reg': 30, 'class_weights': [1, 4]}, 'weighted_2': {'iterations': 1000, 'depth': 7, 'learning_rate': 0.03, 'l2_leaf_reg': 30, 'class_weights': [1, 2]}}
